# 2. ETF Screening and Selection

## Overview
This notebook implements the ETF screening and selection process, focusing on:
1. Filtering for distributing ETFs only
2. Platform availability verification
3. Initial ranking based on TER and dividends
4. Risk-adjusted returns analysis

## Setup and Data Loading

In [1]:
import pandas as pd
import numpy as np
import os
import requests
import json
from datetime import datetime, timedelta
from scipy import stats

## Platform Availability Check
First, we'll define a function to check if ETFs are available on our chosen platform (InvestEngine):

In [2]:
def check_etf_availability(etf_name, investengine_url='https://investengine.com/api/v0.29/public/securities/'):
    '''Check if an ETF is available on InvestEngine.'''
    try:
        search_url = f'{investengine_url}?search={etf_name.replace(" ", "%20")}'
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }
        response = requests.get(search_url, headers=headers)
        response.raise_for_status()
        
        if response.status_code == 304:
            return False
        json_response = response.json()
        
        return isinstance(json_response, list) and len(json_response) > 0
    except Exception as e:
        print(f'Error checking {etf_name}: {e}')
        return False

In [3]:
#Calculate the total returns for benchmarks in 2024
import requests

def get_adjusted_prices(symbol='veve.lon'):
    print(f'Getting adjusted prices for {symbol}')
    url = f'https://www.alphavantage.co/query?function=TIME_SERIES_DAILY_ADJUSTED&symbol={symbol}&outputsize=full&apikey=1JYCUCSEKO7IYQKU'  
    r = requests.get(url)
    data = r.json()
    df = pd.DataFrame(data['Time Series (Daily)']).T
    df.index = pd.to_datetime(df.index)
     # Convert to numeric and round
    df['5. adjusted close'] = pd.to_numeric(df['5. adjusted close'], errors='coerce') 
    df['5. adjusted close'] = df['5. adjusted close'].round(2)
    return df.sort_index(ascending=True)

def get_benchmark_last_year_return(symbol='veve.lon'):    
    df = get_adjusted_prices(symbol)
    #filter start date to 2024-01-01 and end date to 2024-12-31
    df = df[(df.index >= '2024-01-01') & (df.index <= '2024-12-31')]    
    
    # Calculate the total return for the last 12 months
    last_day_price = df[(df.index=='2024-12-31')]['5. adjusted close'].values[0]
    first_day_price = df[(df.index=='2024-01-02')]['5. adjusted close'].values[0]
    # Calculate the 12-month return
    ly_return = ((last_day_price - first_day_price) / first_day_price) * 100
    # Round to 2 decimal places and convert to percentage
    ly_return = round(ly_return, 2)
    
    return ly_return

def calculate_volatility(symbol, period=252):
    df = get_adjusted_prices(symbol)
    # Calculate daily returns
    df['daily_return'] = df['5. adjusted close'].pct_change()
    # Calculate annualized volatility
    volatility = df['daily_return'].std() * np.sqrt(period)
    return round(volatility * 100,2)

eq_benchmark_return = get_benchmark_last_year_return('veve.lon')
bnd_benchmark_return = get_benchmark_last_year_return('saaa.lon')
eq_benchmark_volatility = calculate_volatility('veve.lon')
bnd_benchmark_volatility = calculate_volatility('saaa.lon')
print(f'2024 veve return: {eq_benchmark_return}%')
print(f'2024 saaa return: {bnd_benchmark_return}%')
print(f'2024 veve volatility: {eq_benchmark_volatility}%')
print(f'2024 saaa volatility: {bnd_benchmark_volatility}%')

eq_sharpe_ratio = round(eq_benchmark_return / eq_benchmark_volatility,2)
bond_sharpe_ratio = round(bnd_benchmark_return / bnd_benchmark_volatility,2)
print(f'2024 veve sharpe ratio: {eq_sharpe_ratio}')
print(f'2024 saaa sharpe ratio: {bond_sharpe_ratio}')

Getting adjusted prices for veve.lon
Getting adjusted prices for saaa.lon
Getting adjusted prices for veve.lon
Getting adjusted prices for saaa.lon
2024 veve return: 857.74%
2024 saaa return: -5.4%
2024 veve volatility: nan%
2024 saaa volatility: nan%
2024 veve sharpe ratio: nan
2024 saaa sharpe ratio: nan


In [4]:
benchmark_etfs = [
    # Structure: [Asset Class, Region, Country, Primary Ticker, Description]
    
    # Equity Benchmarks - Developed Markets
    ['Equity', 'Developed_AmericasandUK', 'United Kingdom', 'ISF.LON', 'iShares Core FTSE 100 UCITS ETF GBP (Dist)'],
    ['Equity', 'Developed_AmericasandUK', 'United States', 'SPY', 'SPDR S&P 500 ETF Trust'],
    ['Equity', 'Developed_AmericasandUK', 'Canada', 'ZCN.TRT', 'BMO S&P/TSX Capped Composite'],    
    ['Equity', 'Developed_EMEA', 'Germany', 'CS51.LON', 'iShares VII PLC - iShares Core EURO STOXX 50 ETF EUR Acc'],
    ['Equity', 'Developed_EMEA', 'France', 'CS51.LON', 'iShares VII PLC - iShares Core EURO STOXX 50 ETF EUR Acc'],
    ['Equity', 'Developed_EMEA', 'Italy', 'CS51.LON', 'iShares VII PLC - iShares Core EURO STOXX 50 ETF EUR Acc'],
    ['Equity', 'Developed_EMEA', 'Spain', 'CS51.LON', 'iShares VII PLC - iShares Core EURO STOXX 50 ETF EUR Acc'],
    ['Equity', 'Developed_EMEA', 'Netherlands', 'CS51.LON', 'iShares VII PLC - iShares Core EURO STOXX 50 ETF EUR Acc'],
    ['Equity', 'Developed_EMEA', 'Switzerland', 'CSWG.LON', 'Amundi Index Solutions - Amundi MSCI Switzerland'],    
    ['Equity', 'Developed_APAC', 'Japan', 'XDJP.LON', 'Xtrackers Nikkei 225 UCITS ETF 1D'],
    ['Equity', 'Developed_APAC', 'Australia', 'SAUS.LON', 'iShares MSCI Australia UCITS ETF'],    
    # Equity Benchmarks - Emerging Markets
    ['Equity', 'Emerging_Americas', 'Brazil', 'IBZL.LON', 'iShares MSCI Brazil UCITS ETF'],
    ['Equity', 'Emerging_Americas', 'Mexico', 'XMEX.LON', 'iShares MSCI Mexico Capped ETF'],    
    ['Equity', 'Emerging_APACandEMEA', 'China', 'ASHR', 'XTRACKERS HARVEST CSI 300 CHINA A-SHARES ETF'],
    ['Equity', 'Emerging_APACandEMEA', 'India', 'XNIF.LON', 'Xtrackers Nifty 50 Swap UCITS ETF 1C'],
    ['Equity', 'Emerging_APACandEMEA', 'South Korea', 'EWY', 'iShares MSCI South Korea ETF'],    
]

benchmark_df = pd.DataFrame(
    benchmark_etfs, 
    columns=['asset_class', 'region', 'country', 'benchmark_ticker', 'benchmark_description']
)

# Sort the DataFrame
benchmark_df = benchmark_df.sort_values(['asset_class', 'region', 'country'])
benchmark_df

#call the function get_benchmark_last_year_return to get the adjusted prices for each benchmark ETF
benchmark_df['benchmark_2024_Return'] = benchmark_df['benchmark_ticker'].apply(get_benchmark_last_year_return)


Getting adjusted prices for SAUS.LON
Getting adjusted prices for XDJP.LON
Getting adjusted prices for ZCN.TRT
Getting adjusted prices for ISF.LON
Getting adjusted prices for SPY
Getting adjusted prices for CS51.LON
Getting adjusted prices for CS51.LON
Getting adjusted prices for CS51.LON
Getting adjusted prices for CS51.LON
Getting adjusted prices for CS51.LON
Getting adjusted prices for CSWG.LON
Getting adjusted prices for ASHR
Getting adjusted prices for XNIF.LON
Getting adjusted prices for EWY
Getting adjusted prices for IBZL.LON
Getting adjusted prices for XMEX.LON


In [5]:
benchmark_df

,asset_class,region,country,benchmark_ticker,benchmark_description,benchmark_2024_Return
10,Equity,Developed_APAC,Australia,SAUS.LON,iShares MSCI Australia UCITS ETF,2.81
9,Equity,Developed_APAC,Japan,XDJP.LON,Xtrackers Nikkei 225 UCITS ETF 1D,9.43
2,Equity,Developed_AmericasandUK,Canada,ZCN.TRT,BMO S&P/TSX Capped Composite,22.13
0,Equity,Developed_AmericasandUK,United Kingdom,ISF.LON,iShares Core FTSE 100 UCITS ETF GBP (Dist),9.52
1,Equity,Developed_AmericasandUK,United States,SPY,SPDR S&P 500 ETF Trust,25.59
4,Equity,Developed_EMEA,France,CS51.LON,iShares VII PLC - iShares Core EURO STOXX 50 E...,6.74
3,Equity,Developed_EMEA,Germany,CS51.LON,iShares VII PLC - iShares Core EURO STOXX 50 E...,6.74
5,Equity,Developed_EMEA,Italy,CS51.LON,iShares VII PLC - iShares Core EURO STOXX 50 E...,6.74
7,Equity,Developed_EMEA,Netherlands,CS51.LON,iShares VII PLC - iShares Core EURO STOXX 50 E...,6.74
6,Equity,Developed_EMEA,Spain,CS51.LON,iShares VII PLC - iShares Core EURO STOXX 50 E...,6.74


### Fetch previous day close price from alpha vantage

In [6]:
def get_yday_close_price(symbol='veve.lon'):
    print(f'Getting previous day adjusted close price for {symbol}')
    url = f'https://www.alphavantage.co/query?function=TIME_SERIES_DAILY_ADJUSTED&symbol={symbol}&outputsize=full&apikey=1JYCUCSEKO7IYQKU'  
    r = requests.get(url)
    data = r.json()
    df = pd.DataFrame(data['Time Series (Daily)']).T
    df.index = pd.to_datetime(df.index)
    #fetch and return previous day adjusted close price
    df = df.loc[(df.index >= (datetime.now() - timedelta(days=5))) & (df.index <= datetime.now())]
    df = df[['5. adjusted close']].sort_index(ascending=False)
    date = df.index[0].date().strftime('%Y-%m-%d') 
    price = pd.to_numeric(df.iloc[0].values[0], errors='coerce').round(2)
    print(f'Previous day adjusted close price for {symbol} is {price}')
    return date, price

get_yday_close_price()


Getting previous day adjusted close price for veve.lon
Previous day adjusted close price for veve.lon is 81.4


('2025-05-23', np.float64(81.4))

## ETF Screening Process
Now proceed with screening and categorization using pre-determined weights by country

In [7]:
def get_region_category_from_filename(filename):
    '''Extract region_category from filename pattern justetf_class-{asset}_{region_category}.csv'''
    try:
        parts = filename.split('_')
        region_category = parts[2].replace('.csv', '')
        return region_category.replace('_', ' ')
    except:
        return 'Unknown'

def get_asset_class_from_filename(filename):
    '''Extract asset class from filename pattern justetf_class-{asset}_{region_category}.csv'''
    try:
        parts = filename.split('_')
        asset_class = parts[1]
        return asset_class.replace('class-', '')
    except:
        return 'Unknown'

# Initialize main DataFrame with required columns
main_df_equities = pd.DataFrame()#columns=['region', 'country', 'region_category', 'ticker', 'name', 'ter', 'last_year_dividends', 'last_year_return_per_risk', 'beta'])

# Process ETF files
for csvl in os.listdir('.'):
    if csvl.startswith('justetf_class-equity') and csvl.endswith('.csv'):
        try:
            # Determine asset class and region
            asset_class = get_asset_class_from_filename(csvl)
            region_category = get_region_category_from_filename(csvl)

            #only for equities asset class            
            if not asset_class or region_category == 'Unknown' or asset_class != 'equity':
                print(f'Skipping {csvl} due to asset class or region category mismatch.')
                continue
                
            print(f'Processing {asset_class} file for {region_category}: {csvl}')
            
            # Load and process CSV
            try:
                csv_df = pd.read_csv(csvl)
                if csv_df.empty:
                    print(f'Empty file: {csvl}')
                    continue
            except pd.errors.EmptyDataError:
                print(f'Empty file: {csvl}')
                continue
            
            csv_df['asset_class'] = asset_class
            csv_df['region'] = csv_df['region'].fillna('N/A')
            csv_df['country'] = csv_df['country'].fillna('N/A')
           
            distributing_df = csv_df.copy()
            
            distributing_df = distributing_df[distributing_df['dividends'] == 'Distributing']

            # ensure ets with size greater than 100
            distributing_df = distributing_df[distributing_df['size'] > 100]

            # ensure etfs with hedge = FALSE to reduce costs
            distributing_df = distributing_df[distributing_df['hedged'] == False]

            include_tickers = ['IBZL']
            # ensure etfs with TER < 0.5 except for the include_tickers
            distributing_df = distributing_df[(distributing_df['ter'] < 0.5) | (distributing_df['ticker'].isin(include_tickers))]

            remove_tickers = [
                'XUCN'#no price in alphavantage
                ,'LYXIB'#no price in alphavantage
                ,'C001'#C001.FRK
                ,'SW2CHA' # no price in alphavantage
                ,'XSMI' #XSMI.DEX
                ,'SJPD' #SJPD.AMS
                ,'XESD' #XESD.FRK
                ,'C024'#C024.FRK
                ]
            
            
            
            distributing_df = distributing_df[~distributing_df['ticker'].isin(remove_tickers)]

            #Sort by highest to lowest last_year_dividends for each category and set the intra_region_category as High Yield
            hy_df = distributing_df.copy().sort_values(by=['last_year_dividends'], ascending=False).drop_duplicates(subset=['region_category'])
            hy_df['intra_region_category'] = 'High Yield'

            #join benchmark_df to distributing_df on country, add benchmark_ticker , benchmark_description, benchmark_return
            
            benchmark_distributing_df = distributing_df.copy()[~distributing_df['country'].isin(hy_df['country'])]
            # remove a list of tickers from main_df_equities
            

       
            
            benchmark_distributing_df = pd.merge(benchmark_distributing_df, benchmark_df[['country', 'benchmark_ticker', 'benchmark_description', 'benchmark_2024_Return']], left_on='country', right_on='country', how='left')
            
            benchmark_distributing_df["beta"] = benchmark_distributing_df.apply(lambda x: x["2024"]/x["benchmark_2024_Return"], axis=1)
            benchmark_distributing_df.to_csv('benchmark_distributing_df'+csvl+'.csv', index=False)
            #Ensure beta greater than or equal to 1
            benchmark_distributing_df = benchmark_distributing_df[benchmark_distributing_df['beta'] >= 1]
            beta_df = benchmark_distributing_df.copy().sort_values(by=['ter','beta'], ascending=[True, False]).drop_duplicates(subset=['region_category'])                
            beta_df['intra_region_category'] = 'Beta'
            
            
            main_df_equities = pd.concat([main_df_equities, hy_df, beta_df], ignore_index=True)
            
                      
           

        except Exception as e:
            print(f'Error processing {csvl}: {e}')

#Add a column to check if its available on InvestEngine
main_df_equities['available_on_investengine'] = main_df_equities['ticker'].apply(lambda x: check_etf_availability(x))


main_df_equities[['yday_close_price_date', 'yday_close_price']] = main_df_equities['ticker'].apply(lambda x: get_yday_close_price(x +'.lon')).to_list()
#save main_df to summary.csv
main_df_equities.to_csv('summary_equities.csv', index=False)




Processing equity file for developed: justetf_class-equity_developed_apac.csv
Processing equity file for developed: justetf_class-equity_developed_americasanduk.csv
Processing equity file for developed: justetf_class-equity_developed_emea.csv
Processing equity file for emerging: justetf_class-equity_emerging_apacandemea.csv
Processing equity file for emerging: justetf_class-equity_emerging_americas.csv
Getting previous day adjusted close price for AUAD.lon
Previous day adjusted close price for AUAD.lon is 1827.0
Getting previous day adjusted close price for PRIJ.lon
Previous day adjusted close price for PRIJ.lon is 2368.25
Getting previous day adjusted close price for QYLP.lon
Previous day adjusted close price for QYLP.lon is 11.45
Getting previous day adjusted close price for LCUK.lon
Previous day adjusted close price for LCUK.lon is 12.36
Getting previous day adjusted close price for IMIB.lon
Previous day adjusted close price for IMIB.lon is 2012.75
Getting previous day adjusted clos

In [8]:
#display etf ticker and name grouped by region_category and intra_region_category and show all rows, sort by intra_region_category 
main_df_equities.groupby(['asset_class','region_category', 'intra_region_category'])[['ticker', 'name','country','ter','beta','yday_close_price','last_five_years','last_year_dividends','available_on_investengine','yday_close_price_date', 'yday_close_price']].last().reset_index().sort_values(by=['asset_class', 'intra_region_category'], ascending=[True, True])

,asset_class,region_category,intra_region_category,ticker,name,country,ter,beta,yday_close_price,last_five_years,last_year_dividends,available_on_investengine,yday_close_price_date,yday_close_price
0,equity,Developed_APAC,Beta,PRIJ,Amundi Prime Japan UCITS ETF DR (D),Japan,0.05,1.092259,2368.25,43.51,1.87,True,2025-05-23,2368.25
2,equity,Developed_AmericasandUK,Beta,LCUK,Amundi UK Equity All Cap UCITS ETF Dist,United Kingdom,0.04,1.030462,12.36,71.43,3.74,True,2025-05-23,12.36
4,equity,Developed_EMEA,Beta,XDDX,Xtrackers DAX ESG Screened UCITS ETF 1D,Germany,0.09,1.606825,12650.0,95.61,2.81,True,2025-05-23,12650.0
1,equity,Developed_APAC,High Yield,AUAD,UBS ETF (IE) MSCI Australia UCITS ETF (AUD) A-dis,Australia,0.40,NaN,1827.0,71.76,3.58,True,2025-05-23,1827.0
3,equity,Developed_AmericasandUK,High Yield,QYLP,Global X Nasdaq 100 Covered Call UCITS ETF D,United States,0.45,NaN,11.45,NaN,11.72,True,2025-05-23,11.45
5,equity,Developed_EMEA,High Yield,IMIB,iShares FTSE MIB UCITS ETF EUR (Dist),Italy,0.35,NaN,2012.75,157.90,4.33,True,2025-05-23,2012.75
6,equity,Emerging_APACandEMEA,High Yield,HMCH,HSBC MSCI China UCITS ETF USD,China,0.28,NaN,554.62,-8.95,2.67,True,2025-05-23,554.62
7,equity,Emerging_Americas,High Yield,IBZL,iShares MSCI Brazil UCITS ETF (Dist),Brazil,0.74,NaN,1640.0,55.80,5.59,True,2025-05-23,1640.0


In [9]:

#############BONDS###############
main_df_bonds = pd.DataFrame(columns=['region', 'country', 'region_category', 'ticker', 'name', 'ter', 'last_year_dividends', 'last_year_return_per_risk', 'beta'])
import pandas as pd
data = {
    'asset_class': ['bonds', 'bonds', 'bonds', 'bonds', 'bonds', 'bonds', 'bonds', 'bonds', 'bonds'],
    'region_category': ['Developed_AmericasandUK', 'Developed_AmericasandUK', 'Developed_AmericasandUK', 'Developed_AmericasandUK', 'Developed_EMEA', 'Developed_EMEA', 'Developed_EMEA', 'Emerging_APACandEMEA', 'Emerging_APACandEMEA'],
    'intra_region_category': ['Govt', 'Corp', 'Govt', 'Corp', 'Govt', 'Corp', 'High yield', 'Corp', 'Govt'],
    'ticker': ['IGLT', 'SLXX', 'TRXG', 'UC81', 'PRIR', 'VECP', 'JNKE', 'EMCP', 'VEMT']
}
#No UC84 in investengine, so going with UC81


df_bonds_list = pd.DataFrame(data)
#remove JNKE from df_bonds_list not available on investengine :(
df_bonds_list = df_bonds_list[df_bonds_list['ticker'] != 'JNKE']

for csvl in os.listdir('.'):
    if csvl.startswith('justetf_class-bonds_') and csvl.endswith('.csv'):
        try:
            asset_class = get_asset_class_from_filename(csvl)
            region_category = get_region_category_from_filename(csvl)
            
            print(f'Processing {asset_class} file for {region_category}: {csvl}')
            
            # Load and process CSV
            try:
                csv_df = pd.read_csv(csvl)
                if csv_df.empty:
                    print(f'Empty file: {csvl}')
                    continue
            except pd.errors.EmptyDataError:
                print(f'Empty file: {csvl}')
                continue

            csv_df['asset_class'] = asset_class
            csv_df['region'] = csv_df['region'].fillna('N/A')
            csv_df['country'] = csv_df['country'].fillna('N/A')
        
            #check if ticker in df_bonds_list is present in csv_df
            for index, row in df_bonds_list.iterrows():
                if row['ticker'] in csv_df['ticker'].values:
                    ticker = row['ticker']
                    print(f'Processing {ticker} in {csvl}')
                    # Filter the csv_df for the current ticker
                    csv_df_ticker = csv_df[csv_df['ticker'] == ticker]  
                    csv_df_ticker['intra_region_category'] = row['intra_region_category']
                    csv_df_ticker['region_category'] = row['region_category']
                    # Append to main_df                   
                    main_df_bonds = pd.concat([main_df_bonds, csv_df_ticker], ignore_index=True)

        except Exception as e:
            print(f'Error processing {csvl}: {e}')
            continue



#Add a column to check if its available on InvestEngine
main_df_bonds['available_on_investengine'] = main_df_bonds['ticker'].apply(lambda x: check_etf_availability(x))
main_df_bonds[['yday_close_price_date', 'yday_close_price']] = main_df_bonds['ticker'].apply(lambda x: get_yday_close_price(x +'.lon')).to_list()
#save main_df to summary.csv
main_df_bonds.to_csv('summary_bonds.csv', index=False)
             



Processing bonds file for developed: justetf_class-bonds_developed_apac.csv
Processing bonds file for developed: justetf_class-bonds_developed_americasanduk.csv
Processing IGLT in justetf_class-bonds_developed_americasanduk.csv
Processing SLXX in justetf_class-bonds_developed_americasanduk.csv
Processing TRXG in justetf_class-bonds_developed_americasanduk.csv
Processing UC81 in justetf_class-bonds_developed_americasanduk.csv
Processing EMCP in justetf_class-bonds_developed_americasanduk.csv
Processing VEMT in justetf_class-bonds_developed_americasanduk.csv
Processing bonds file for developed: justetf_class-bonds_developed_emea.csv
Processing PRIR in justetf_class-bonds_developed_emea.csv
Processing VECP in justetf_class-bonds_developed_emea.csv
Processing bonds file for emerging: justetf_class-bonds_emerging_apacandemea.csv
Processing bonds file for emerging: justetf_class-bonds_emerging_americas.csv
Empty file: justetf_class-bonds_emerging_americas.csv


C:\Users\rakes\AppData\Local\Temp\ipykernel_60260\3943162324.py:46: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  csv_df_ticker['intra_region_category'] = row['intra_region_category']
C:\Users\rakes\AppData\Local\Temp\ipykernel_60260\3943162324.py:47: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  csv_df_ticker['region_category'] = row['region_category']
C:\Users\rakes\AppData\Local\Temp\ipykernel_60260\3943162324.py:49: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is d

Getting previous day adjusted close price for IGLT.lon
Previous day adjusted close price for IGLT.lon is 9.7
Getting previous day adjusted close price for SLXX.lon
Previous day adjusted close price for SLXX.lon is 120.74
Getting previous day adjusted close price for TRXG.lon
Previous day adjusted close price for TRXG.lon is 2611.25
Getting previous day adjusted close price for UC81.lon
Previous day adjusted close price for UC81.lon is 1024.75
Getting previous day adjusted close price for EMCP.lon
Previous day adjusted close price for EMCP.lon is 66.74
Getting previous day adjusted close price for VEMT.lon
Previous day adjusted close price for VEMT.lon is 31.1
Getting previous day adjusted close price for PRIR.lon
Previous day adjusted close price for PRIR.lon is 1453.7
Getting previous day adjusted close price for VECP.lon
Previous day adjusted close price for VECP.lon is 40.88


In [10]:
summary_df = pd.concat([main_df_equities, main_df_bonds], ignore_index=True)
summary_df.to_csv('summary_all.csv', index=False)
summary_df

C:\Users\rakes\AppData\Local\Temp\ipykernel_60260\913047572.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  summary_df = pd.concat([main_df_equities, main_df_bonds], ignore_index=True)


,wkn,ticker,valor,name,inception_date,age_in_days,age_in_years,strategy,domicile_country,currency,...,region_category,asset_class,intra_region_category,benchmark_ticker,benchmark_description,benchmark_2024_Return,beta,available_on_investengine,yday_close_price_date,yday_close_price
0,A2DWP8,AUAD,22896454.0,UBS ETF (IE) MSCI Australia UCITS ETF (AUD) A-dis,2017-09-18 00:00:00,2776.0,7.605479,Long-only,Ireland,AUD,...,Developed_APAC,equity,High Yield,NaN,NaN,NaN,NaN,True,2025-05-23,1827.0
1,A2PBLK,PRIJ,45676821.0,Amundi Prime Japan UCITS ETF DR (D),2019-01-30,2277.0,6.238356,Long-only,Luxembourg,JPY,...,Developed_APAC,equity,Beta,XDJP.LON,Xtrackers Nikkei 225 UCITS ETF 1D,9.43,1.092259,True,2025-05-23,2368.25
2,A2QR39,QYLP,122393230.0,Global X Nasdaq 100 Covered Call UCITS ETF D,2022-11-22,885.0,2.424658,Long-only,Ireland,USD,...,Developed_AmericasandUK,equity,High Yield,NaN,NaN,NaN,NaN,True,2025-05-23,11.45
3,LYX0YA,LCUK,40587057.0,Amundi UK Equity All Cap UCITS ETF Dist,2018-02-27 00:00:00,2614.0,7.161644,Long-only,Luxembourg,GBP,...,Developed_AmericasandUK,equity,Beta,ISF.LON,iShares Core FTSE 100 UCITS ETF GBP (Dist),9.52,1.030462,True,2025-05-23,12.36
4,A0MZWP,IMIB,3246482.0,iShares FTSE MIB UCITS ETF EUR (Dist),2007-07-06 00:00:00,6503.0,17.816438,Long-only,Ireland,EUR,...,Developed_EMEA,equity,High Yield,NaN,NaN,NaN,NaN,True,2025-05-23,2012.75
5,DBX0NH,XDDX,20028815.0,Xtrackers DAX ESG Screened UCITS ETF 1D,2012-11-28,4531.0,12.413699,Long-only,Luxembourg,EUR,...,Developed_EMEA,equity,Beta,CS51.LON,iShares VII PLC - iShares Core EURO STOXX 50 E...,6.74,1.606825,True,2025-05-23,12650.0
6,A1JHYT,HMCH,12534159.0,HSBC MSCI China UCITS ETF USD,2011-01-26,5203.0,14.254795,Long-only,Ireland,USD,...,Emerging_APACandEMEA,equity,High Yield,NaN,NaN,NaN,NaN,True,2025-05-23,554.62
7,A0HGWA,IBZL,2308866.0,iShares MSCI Brazil UCITS ETF (Dist),2005-11-18,7098.0,19.446575,Long-only,Ireland,USD,...,Emerging_Americas,equity,High Yield,NaN,NaN,NaN,NaN,True,2025-05-23,1640.0
8,A0LGP9,IGLT,2803931.0,iShares Core UK Gilts UCITS ETF,2006-12-01 00:00:00,6720.0,18.410959,Long-only,Ireland,GBP,...,Developed_AmericasandUK,bonds,Govt,NaN,NaN,NaN,NaN,True,2025-05-23,9.7
9,A0DKL3,SLXX,1828476.0,iShares Core GBP Corporate Bond UCITS ETF,2004-03-29 00:00:00,7697.0,21.087671,Long-only,Ireland,GBP,...,Developed_AmericasandUK,bonds,Corp,NaN,NaN,NaN,NaN,True,2025-05-23,120.74


In [ ]:
#print in column orientation
main_df[main_df['ticker']=='MLPP'].transpose()

#get_monthly_market_returns('IS0P.EUR')

,3
region,AmericasandUK
country,United States
region_category,Developed_AmericasandUK
ticker,MLPP
name,Invesco Morningstar US Energy Infrastructure M...
ter,0.5
last_year_dividends,7.8
last_year_return_per_risk,0.15
beta,NaN
wkn,A1T96S


{'bestMatches': [{'1. symbol': 'XDGM.FRK', '2. name': 'db x-trackers Mittelstand & MidCap Germany UCITS DR 1D', '3. type': 'ETF', '4. region': 'Frankfurt', '5. marketOpen': '08:00', '6. marketClose': '20:00', '7. timezone': 'UTC+02', '8. currency': 'EUR', '9. matchScore': '0.8000'}, {'1. symbol': 'XDGM.DEX', '2. name': 'db x-trackers Mittelstand & MidCap Germany UCITS DR 1D', '3. type': 'ETF', '4. region': 'XETRA', '5. marketOpen': '08:00', '6. marketClose': '20:00', '7. timezone': 'UTC+02', '8. currency': 'EUR', '9. matchScore': '0.7273'}]}


In [ ]:
 if len(distributing_df) == 0:
                print(f'No distributing ETFs found in {csvl}')
                continue
            
            # Calculate beta using appropriate benchmark
            distributing_df['beta'] = distributing_df.apply(
                lambda x: calculate_beta(x.get('returns', []), x['Asset_Class']),
                axis=1
            )

            # Calculate yield statistics by region
            # Collect all yields for each region in a list
            region_yields = {}
            for region_name in distributing_df['Region'].unique():
                region_yields[region_name] = distributing_df[
                    distributing_df['Region'] == region_name
                ]['last_year_dividends'].dropna().tolist()

            # Assign region yields to each row
            distributing_df['group_yields'] = distributing_df['Region'].map(region_yields)

            # Categorize ETFs
            distributing_df['category'] = distributing_df.apply(categorize_etf, axis=1)

            # Sort within each region and category
            sorted_df = distributing_df.sort_values(
                by=['Region', 'category', 'ter', 'last_year_return_per_risk'],
                ascending=[True, True, True, False]
            )

            # Select columns for grouping
            cols_to_keep = ['Region', 'category', 'ticker', 'name', 'ter', 'last_year_dividends', 'last_year_return_per_risk', 'beta']

            # Select top ETF from each region-category combination
            top_etfs = sorted_df.groupby(['Region', 'category'], as_index=False).first()[cols_to_keep]

            # Add to main DataFrame
            main_df = pd.concat([main_df, top_etfs], ignore_index=True)
if not main_df.empty:
    # Check platform availability
    main_df['investengine'] = main_df['ticker'].apply(check_etf_availability)

    # Filter for available ETFs
    final_etfs = main_df[main_df['investengine']].copy()

    # Save results with categories and ticker information
    final_etfs.to_csv('screened_etfs_categorized.csv', index=False)

    # Display results grouped by region and category, showing ticker
    display(final_etfs.pivot_table(
        index='Region',
        columns='category',
        values='ticker',
        aggfunc='first'
    ))

    # Run analysis on final screened ETFs instead of all distributing ETFs
    analyze_betas(main_df)
else:
    print('No ETFs found matching the criteria')

In [ ]:
get_monthly_market_returns('PRIJ.LON')


C:\Users\rakes\AppData\Local\Temp\ipykernel_34780\1573099026.py:35: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df['return'] = df['return'].resample('M').agg(lambda x: (1 + x).prod() - 1)


2025-03-31    0.039709
2025-02-28   -0.001505
2025-01-31   -0.011537
2024-12-31    0.027603
2024-10-31    0.024132
2024-09-30    0.008136
2024-07-31    0.002732
2024-05-31   -0.006690
2024-04-30    0.029772
2024-02-29   -0.056837
2024-01-31   -0.032830
2023-11-30   -0.009056
2023-10-31    0.004657
2023-08-31   -0.002152
2023-07-31   -0.003847
2023-06-30   -0.021298
2023-05-31   -0.035332
2023-03-31   -0.017167
2023-02-28    0.015918
2023-01-31   -0.033289
2022-11-30   -0.020519
2022-10-31    0.005431
2022-09-30    0.036141
2022-08-31    0.013468
2022-06-30    0.045756
2022-05-31   -0.014730
2022-03-31   -0.004768
2022-02-28    0.006511
2022-01-31    0.059254
2021-12-31   -0.004609
2021-11-30    0.023035
2021-09-30   -0.016371
2021-08-31   -0.028272
2021-06-30   -0.017081
2021-04-30    0.028051
2021-03-31   -0.003848
2020-12-31   -0.012472
2020-11-30   -0.052448
2020-09-30   -0.056862
2020-07-31    0.029604
2020-06-30    0.013065
2020-04-30   -0.056897
2020-03-31    0.086297
2020-01-31 